# 🤖 Notebook 03 — Model Training & Evaluation
Trains LR, Random Forest, XGBoost and compares their performance.
---

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_score, recall_score, f1_score, roc_auc_score
)

from data_preprocessing import preprocess
from feature_engineering import prepare_data

print('All imports successful ✓')

## 1. Load & Prepare Data

In [ ]:
df = preprocess(path='../data/raw/creditcard.csv')
X_train, X_test, y_train, y_test = prepare_data(df)
feature_names = list(df.drop(columns=['Class']).columns)

print(f'Train size: {len(X_train):,}  |  Test size: {len(X_test):,}')

## 2. Train All Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced'),
    'XGBoost':             XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6,
                                         random_state=42, eval_metric='logloss', use_label_encoder=False, n_jobs=-1),
}

results = []
trained = {}

for name, model in models.items():
    print(f'Training {name} ...')
    model.fit(X_train, y_train)
    trained[name] = model

    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results.append({
        'Model':     name,
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1':        round(f1_score(y_test, y_pred, zero_division=0), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_proba), 4),
    })

results_df = pd.DataFrame(results)
print('\n=== Model Comparison ===')
print(results_df.to_string(index=False))

## 3. Visual Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results_df))
w = 0.2
for i, metric in enumerate(['Precision','Recall','F1','ROC-AUC']):
    ax.bar(x + i*w, results_df[metric], w, label=metric)
ax.set_xticks(x + w*1.5)
ax.set_xticklabels(results_df['Model'])
ax.set_ylim(0, 1.1)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Best Model — Detailed Evaluation

In [ ]:
best_name  = results_df.loc[results_df['F1'].idxmax(), 'Model']
best_model = trained[best_name]
print(f'Best model: {best_name}')

y_pred  = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Normal','Fraud']))

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal','Fraud'],
            yticklabels=['Normal','Fraud'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=13)
plt.tight_layout()
plt.show()

## 6. ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for name, model in trained.items():
    y_p = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_p)
    roc = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC={roc:.4f})')

ax.plot([0,1],[0,1],'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

## 7. SHAP Feature Importance

In [ ]:
import shap

# Sample 500 rows for speed
X_sample = pd.DataFrame(X_test, columns=feature_names).sample(500, random_state=42)

explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_sample)

# For binary classifiers, take class-1 (fraud) SHAP values
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

shap.summary_plot(sv, X_sample, feature_names=feature_names, plot_type='bar', show=True)

## 8. Save Best Model

In [ ]:
import json, os
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/fraud_model.pkl')
with open('../models/feature_names.json','w') as f:
    json.dump(feature_names, f)
results_df.to_csv('../models/model_metrics.csv', index=False)
print(f'Model saved: {best_name}')
print('Metrics saved to ../models/model_metrics.csv')